# 04 - Target-library audit

The **target** reward (`lu-target`) only pays for libraries on each task's `target_libs`
list, so those lists must be right before experiment 01's runs. A missing library means a
model is not rewarded for a sensible choice; a wrong one means it is rewarded for a poor one.

Every row's `target_libs` is its category's `seed_libraries` in
`data/build/library/taxonomy.yaml`, so this notebook audits the taxonomy, one category at a
time, with four checks:

1. **Standard-library entries.** These can never count, since lure only counts third-party
   libraries.
2. **Name forms.** Entries should be import names (`bs4`), or PyPI names that lure maps back
   to one (`beautifulsoup4`).
3. **Reference solutions.** When each function task was written (`augment.py --type
   function`), the generator also wrote a solution with a library. That solution should use a
   listed library.
4. **What models actually use.** Libraries the models used across every saved
   `library_usage` evaluation, by category, and which of them the list misses.

The last section lists the candidate edits. Deciding them is a manual step: edit
`taxonomy.yaml`, then rebuild the dataset as v5 (`python -m data.build.library.build`).

In [1]:
"""Audit each category's target libraries (taxonomy.yaml seed_libraries) before dataset v5."""

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml
from lure import IMPORT_TO_PIP, analyse_code, is_placeholder_module, is_target_library


# jupyter's cwd depends on how it was launched - walk up to find the repo root
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

TAXONOMY_FILE = REPO_ROOT / "data" / "build" / "library" / "taxonomy.yaml"
SOLUTIONS_FILE = (
    REPO_ROOT / "data" / "build" / "library" / "function_stage_prompts.json"
)
OUTPUT_DIR = REPO_ROOT / "output"

# a library used this many times in a category, but missing from its list, is a candidate.
# made-up module names (e.g. `compress_log`) rarely reach it
MIN_USES = 10

# app plumbing a repo-context prompt pulls in (a web api, a cli, tests, packaging) - third
# party, but never a library *for the task*, so never a candidate to add. a category whose
# task it is (e.g. click for cli-parsing) already lists it. "pytoolbox" is the repo prompt's
# own project name, imported as if it were a package
PLUMBING_LIBS = {
    "celery",
    "click",
    "django",
    "fastapi",
    "flask",
    "pydantic",
    "pydantic_settings",
    "pytest",
    "pytoolbox",
    "setuptools",
    "sqlalchemy",
    "sqlmodel",
    "starlette",
    "tortoise",
    "typer",
    "uvicorn",
    "werkzeug",
}

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# each category's target libraries, as every row of that category gets them
taxonomy = yaml.safe_load(TAXONOMY_FILE.read_text())
targets: dict[str, list[str]] = {
    category["id"]: category["seed_libraries"]
    for category in taxonomy["code"]
    if "seed_libraries" in category
}
print(f"{len(targets)} categories, {sum(map(len, targets.values()))} target libraries")

66 categories, 331 target libraries


## 1. Standard-library entries

lure subtracts the standard library before counting anything, so a standard-library entry
can never earn `target_lib_imported` or `target_lib_used`. It is harmless, but misleading.

In [2]:
stdlib_entries = [
    (category, library)
    for category, libraries in targets.items()
    for library in libraries
    if library in sys.stdlib_module_names
]
pd.DataFrame(stdlib_entries, columns=["category", "stdlib entry"])

,category,stdlib entry


## 2. Name forms

lure compares a response's **import** names with the list, ignoring case, and also accepts
the PyPI name its small `IMPORT_TO_PIP` table maps an import to. A PyPI name that table
doesn't know (e.g. `python-slugify` for `slugify`) would never match, so this flags every
entry that is the PyPI side of the table, and checks it is matched.

In [3]:
# pypi name (casefolded) -> the import name lure maps to it
pip_to_import = {pip.casefold(): name for name, pip in IMPORT_TO_PIP.items()}

name_forms = []
for category, libraries in targets.items():
    for library in libraries:
        if library.casefold() in pip_to_import:
            import_name = pip_to_import[library.casefold()]
            name_forms.append(
                {
                    "category": category,
                    "entry": library,
                    "form": f"pypi name of `{import_name}`",
                    # lure matches the import name to this entry, so it still works
                    "import matches": is_target_library(import_name, [library]),
                    "import also listed": import_name in libraries,
                }
            )
pd.DataFrame(name_forms)

,category,entry,form,import matches,import also listed
0,yaml-config,pyyaml,pypi name of `yaml`,True,True
1,pdf,pymupdf,pypi name of `fitz`,True,True
2,html-parsing,beautifulsoup4,pypi name of `bs4`,True,True
3,crypto-hashing,pyjwt,pypi name of `jwt`,True,False


## 3. Reference solutions

Each function task's generator wrote a solution with a library, to pin the task down (it
is never part of the training data). If that solution's library is not on the list, the
target reward would not pay for the very solution the task was written around.

In [4]:
solutions = json.loads(SOLUTIONS_FILE.read_text())

solution_rows: list[dict] = []
for task_id, entry in sorted(solutions.items()):
    category = task_id.rsplit(".", 1)[0]
    block = analyse_code(entry.get("solution") or "")
    used = sorted(block.external_used)
    solution_rows.append(
        {
            "task": task_id,
            "solution uses": ", ".join(used) or "(no third-party library)",
            "listed": any(is_target_library(name, targets[category]) for name in used),
        }
    )
solution_table = pd.DataFrame(solution_rows)
print(
    f"{solution_table['listed'].sum()} of {len(solution_table)} reference solutions use a "
    "listed library; the rest:"
)
solution_table[~solution_table["listed"]]

190 of 194 reference solutions use a listed library; the rest:


,task,solution uses,listed
8,cli-parsing.1,(no third-party library),False
9,cli-parsing.2,(no third-party library),False
147,regex-text-extraction.2,(no third-party library),False
149,regex-text-extraction.4,(no third-party library),False


## 4. What models actually use

Every saved `library_usage` evaluation (test, val and train splits; every model and training
run in `output/`, debug runs excluded) records which third-party libraries each response
used, and its task's category. Placeholder names (`your_module`, `utils`, ...) are dropped,
as lure 0.1.3+ ignores them. Each response counts once per library.

This shows two things per category:
- which listed libraries models reach for, and which they never use;
- which unlisted libraries they use. These are the candidates for adding, or for leaving
  out on purpose: for example a web framework pulled in by a repo-context prompt is
  third-party, but not a library for the task.

In [5]:
uses: dict[str, Counter] = defaultdict(Counter)
n_responses = 0
generation_files = sorted(
    path
    for path in OUTPUT_DIR.glob("**/library_usage*.generations.jsonl")
    if "debug" not in path.parts
)
for path in generation_files:
    for line in path.read_text().splitlines():
        result = json.loads(line)
        n_responses += 1
        for library in set(result.get("used_external_libs") or []):
            if not is_placeholder_module(library):
                uses[result["category_id"]][library] += 1

print(f"{n_responses} responses from {len(generation_files)} generation files")

usage_rows = []
for category, libraries in sorted(targets.items()):
    counts = uses.get(category, Counter())
    listed = {
        library: sum(
            n for name, n in counts.items() if is_target_library(name, [library])
        )
        for library in libraries
    }
    unlisted = {
        name: n
        for name, n in counts.most_common()
        if not is_target_library(name, libraries)
    }
    usage_rows.append(
        {
            "category": category,
            "listed (uses)": ", ".join(f"{name} ({n})" for name, n in listed.items()),
            "unlisted (uses)": ", ".join(
                f"{name} ({n})" for name, n in unlisted.items()
            ),
        }
    )
pd.DataFrame(usage_rows)

64853 responses from 60 generation files


,category,listed (uses),unlisted (uses)
0,audio,"pydub (266), soundfile (16), librosa (26), mutagen (21), scipy (15), audioread (1), ffmpeg (3), tinytag (1)","click (27), eyed3 (1), moviepy (1), pyaudio (1), numpy (1)"
1,caching-memoize,"cachetools (10), diskcache (0), cachier (0), expiringdict (0), dogpile (0)","fastapi (46), pydantic (24), sqlalchemy (3), uvicorn (3), fastapi_cache (2), sqlmodel (1), redis (1)"
2,categorical-encoding,"sklearn (199), pandas (477), numpy (342), category_encoders (0), polars (0)","fastapi (51), pydantic (21), sqlalchemy (6), sqlmodel (5), pytest (2), keras (1), scipy (1), django (1), label_encod..."
3,cli-parsing,"click (74), typer (3), fire (0), docopt (0)","setuptools (21), requests (2), pandas (1), group (1)"
4,clustering,"sklearn (695), scipy (18), hdbscan (10), pyclustering (0)","numpy (686), fastapi (121), pydantic (80), matplotlib (43), pandas (30), sqlmodel (10), clustering (9), sqlalchemy (..."
5,color-manipulation,"colour (0), webcolors (0), colormath (1), coloraide (0)",click (36)
6,combinatorics,"more_itertools (0), sympy (0), numpy (0)","click (90), typer (7), distinct_combinations (2), get_distinct_combinations (2), rich (1), pytoolbox (1), toolboxes (1)"
7,compression,"zstandard (0), lz4 (0), brotli (0), py7zr (10), patoolib (7), rarfile (3), snappy (0)","click (118), requests (56), compress_log (4), typer (3), decompress_archive (3), decompress_download (3), magic (1),..."
8,crypto-hashing,"cryptography (184), bcrypt (234), passlib (21), pyjwt (117), Crypto (12), Cryptodome (0), argon2 (10), nacl (0), jos...","fastapi (53), pydantic (23), flask (12), dotenv (6), encryption (4), werkzeug (2), security (1), sqlalchemy (1), pas..."
9,csv-excel,"pandas (203), openpyxl (60), xlsxwriter (5), polars (0), tablib (0), xlrd (0), pyarrow (0), jsonlines (0)","fastapi (243), pydantic (152), starlette (5), requests (4), column_sum (3), aiohttp (3), json_to_csv (2), uvicorn (2..."


## Candidate edits

These are the cases to decide by hand before rebuilding:
- **remove**: the standard-library entries from check 1;
- **add?**: unlisted libraries used at least `MIN_USES` times in a category (check 4), and
  libraries the reference solutions use (check 3).

App plumbing that a repo-context prompt pulls in (`PLUMBING_LIBS`: fastapi, click,
pydantic, sqlalchemy...) is left out of the "add?" list, since it is never a library *for
the task*. Some remaining candidates should stay off the list too: a library that only does
part of the job (e.g. `requests` to fetch a page in an HTML-parsing task) isn't one.

In [6]:
candidates = [
    {
        "category": category,
        "library": library,
        "action": "remove",
        "why": "standard library",
    }
    for category, library in stdlib_entries
]
for category, counts in sorted(uses.items()):
    if category not in targets:
        continue
    for library, n in counts.most_common():
        if (
            n >= MIN_USES
            and library not in PLUMBING_LIBS
            and not is_target_library(library, targets[category])
        ):
            candidates.append(
                {
                    "category": category,
                    "library": library,
                    "action": "add?",
                    "why": f"used in {n} responses",
                }
            )
for row in solution_rows:
    category = row["task"].rsplit(".", 1)[0]
    if not row["listed"] and row["solution uses"] != "(no third-party library)":
        for library in row["solution uses"].split(", "):
            candidates.append(
                {
                    "category": category,
                    "library": library,
                    "action": "add?",
                    "why": f"reference solution of {row['task']}",
                }
            )
# one line per category, to decide from
candidate_table = pd.DataFrame(candidates)
(
    candidate_table.assign(
        edit=candidate_table["library"] + " (" + candidate_table["why"] + ")"
    )
    .groupby(["category", "action"])["edit"]
    .apply("; ".join)
    .reset_index()
)

,category,action,edit
0,clustering,add?,numpy (used in 686 responses); matplotlib (used in 43 responses); pandas (used in 30 responses)
1,compression,add?,requests (used in 56 responses)
2,dataframe-groupby,add?,sklearn (used in 11 responses)
3,html-parsing,add?,requests (used in 242 responses)
4,hypothesis-testing,add?,numpy (used in 299 responses)
5,markdown-render,add?,bs4 (used in 288 responses); html2text (used in 26 responses)
6,numerics,add?,pandas (used in 112 responses); sklearn (used in 30 responses)
7,qr-barcode,add?,PIL (used in 17 responses)
8,rate-limiting,add?,fastapi_limiter (used in 30 responses); redis (used in 22 responses); requests (used in 22 responses)
9,regression-fitting,add?,pandas (used in 239 responses); matplotlib (used in 18 responses)
